In [1]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 

def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding)
# -------------------------
def cost_fn(x, y):
    return 0.5 * ((x**4 - 16 * x**2 + 5 * x) + (y**4 - 16 * y**2 + 5 * y))

def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C
def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
        "Complete Graph": nx.complete_graph(n),
        "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
        "Grid": build_grid(rows, cols),
        "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
        "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
        "QMOA Complete": build_qmoa_complete(rows, cols),
        "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
        "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
        "Grid + Complete Diagonals": build_grid_with_complete_diagonals(rows, cols)
    }

    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn, -5, 5, -5, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps (Styblinski-Tang):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")

Minimum Spectral Gaps (Styblinski-Tang):
Complete Graph: 0.009807
Hypercube Q8: 0.010078
Grid: 0.004850
Grid + Diagonals: 0.009314
Grid + Main Diagonals: 0.005402
QMOA Complete: 0.013237
QMOA Complete + local Diagonals: 0.012040
QMOA Complete + Principal Complete Diagonals: 0.001111
Grid + Complete Diagonals: 0.000636
Grid with Parallel Edges: 0.004002


In [3]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y,a=1,b=100):
    return (a - x)**2 + b * (y - x**2)**2


def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-4.9, 5, -4.9, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Rosenbrock)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Rosenbrock)):
Complete Graph: 0.000022
Hypercube Q8: 0.000015
Grid: 0.000022
Grid + Diagonals: 0.000022
Grid + Main Diagonals: 0.000022
QMOA Complete: 0.000022
QMOA Complete + local Diagonals: 0.000022
QMOA Complete + Principal Complete Diagonals: 0.000022
Grid + Complete Diagonals: 0.000003
Grid with Parallel Edges: 0.000022


In [1]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 32, 32
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return 100 * np.sqrt(np.abs(y - 0.01 * x**2)) + 0.01 * np.abs(x + 10)


def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q10": nx.convert_node_labels_to_integers(nx.hypercube_graph(10)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn, -15, -5, -3, 3, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Bukin N6)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Bukin N6)):
Complete Graph: 0.008552
Hypercube Q10: 0.006253
Grid: 0.000613
Grid + Diagonals: 0.000542
Grid + Main Diagonals: 0.000587
QMOA Complete: 0.007920
QMOA Complete + local Diagonals: 0.008515
QMOA Complete + Principal Complete Diagonals: 0.006851
Grid + Complete Diagonals: 0.000264
Grid with Parallel Edges: 0.000613


In [5]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2


def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn, -5, 5, -5, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Himmelblau)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Himmelblau)):
Complete Graph: 0.000770
Hypercube Q8: 0.000005
Grid: 0.000030
Grid + Diagonals: 0.000107
Grid + Main Diagonals: 0.000002
QMOA Complete: 0.000389
QMOA Complete + local Diagonals: 0.000506
QMOA Complete + Principal Complete Diagonals: 0.000403
Grid + Complete Diagonals: 0.000019
Grid with Parallel Edges: 0.000030


In [7]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    term1 = (1.5 - x + x*y)**2
    term2 = (2.25 - x + x*y**2)**2
    term3 = (2.625 - x + x*y**3)**2
    return term1 + term2 + term3


def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-4.4, 4.5, -4.4, 4.5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Beale)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Beale)):
Complete Graph: 0.000002
Hypercube Q8: 0.000002
Grid: 0.000002
Grid + Diagonals: 0.000002
Grid + Main Diagonals: 0.000002
QMOA Complete: 0.000002
QMOA Complete + local Diagonals: 0.000002
QMOA Complete + Principal Complete Diagonals: 0.000002
Grid + Complete Diagonals: 0.000002
Grid with Parallel Edges: 0.000002


In [1]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    term1 = 1 + (x + y + 1)**2 * (19 - 14*x + 3*x**2 - 14*y + 6*x*y + 3*y**2)
    term2 = 30 + (2*x - 3*y)**2 * (18 - 32*x + 12*x**2 + 48*y - 36*x*y + 27*y**2)
    return term1 * term2



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-1.9, 2, -1.9, 2, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Goldstein-Price)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Goldstein-Price)):
Complete Graph: 0.000001
Hypercube Q8: 0.000001
Grid: 0.000001
Grid + Diagonals: 0.000001
Grid + Main Diagonals: 0.000001
QMOA Complete: 0.000001
QMOA Complete + local Diagonals: 0.000001
QMOA Complete + Principal Complete Diagonals: 0.000001
Grid + Complete Diagonals: 0.000001
Grid with Parallel Edges: 0.000001


## Rugged

In [2]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 32, 32
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return  20 + x**2 + y**2 - 10 * (np.cos(2 * np.pi * x) + np.cos(2 * np.pi * y))



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q10": nx.convert_node_labels_to_integers(nx.hypercube_graph(10)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-4.9, 5, -4.9, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Rastrigin):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Rastrigin):
Complete Graph: 0.002708
Hypercube Q10: 0.003209
Grid: 0.000977
Grid + Diagonals: 0.001283
Grid + Main Diagonals: 0.000000
QMOA Complete: 0.003451
QMOA Complete + local Diagonals: 0.003446
QMOA Complete + Principal Complete Diagonals: 0.000042
Grid + Complete Diagonals: 0.000000
Grid with Parallel Edges: 0.000977


In [1]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y,m=10):
    term1 = np.sin(x) * (np.sin((x**2)/np.pi) ** (2 * m))
    term2 = np.sin(y) * (np.sin((2 * y**2)/np.pi) ** (2 * m))
    return -(term1 + term2)



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-5, 5, -5, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Michalewicz):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Michalewicz):
Complete Graph: 0.035950
Hypercube Q8: 0.042552
Grid: 0.004778
Grid + Diagonals: 0.008793
Grid + Main Diagonals: 0.005287
QMOA Complete: 0.072830
QMOA Complete + local Diagonals: 0.069507
QMOA Complete + Principal Complete Diagonals: 0.062826
Grid + Complete Diagonals: 0.004172
Grid with Parallel Edges: 0.003621


In [2]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    term1 = (x**2 + y**2) / 4000
    term2 = np.cos(x / np.sqrt(1)) * np.cos(y / np.sqrt(2))
    return term1 - term2 + 1



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-4.9, 5, -4.9, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Griewank):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Griewank):
Complete Graph: 0.003565
Hypercube Q8: 0.000641
Grid: 0.000001
Grid + Diagonals: 0.000005
Grid + Main Diagonals: 0.000021
QMOA Complete: 0.003751
QMOA Complete + local Diagonals: 0.003746
QMOA Complete + Principal Complete Diagonals: 0.003743
Grid + Complete Diagonals: 0.000030
Grid with Parallel Edges: 0.000008


In [3]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return 0.26 * (x**2 + y**2) - 0.48 * x * y



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-4.9, 5, -4.9, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Matyas):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Matyas):
Complete Graph: 0.000108
Hypercube Q8: 0.000108
Grid: 0.000108
Grid + Diagonals: 0.000108
Grid + Main Diagonals: 0.000108
QMOA Complete: 0.000108
QMOA Complete + local Diagonals: 0.000108
QMOA Complete + Principal Complete Diagonals: 0.000108
Grid + Complete Diagonals: 0.000108
Grid with Parallel Edges: 0.000108


In [4]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    r = np.sqrt(x**2 + y**2)
    return -np.abs(np.sin(x) * np.cos(y) * np.exp(np.abs(1 - r/np.pi)))



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-9.9, 10, -9.9, 10, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Holder Table):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Holder Table):
Complete Graph: 0.010828
Hypercube Q8: 0.001128
Grid: 0.000137
Grid + Diagonals: 0.000250
Grid + Main Diagonals: 0.000060
QMOA Complete: 0.013235
QMOA Complete + local Diagonals: 0.011280
QMOA Complete + Principal Complete Diagonals: 0.013510
Grid + Complete Diagonals: 0.000477
Grid with Parallel Edges: 0.000201


In [5]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return np.sin(x + y) + (x - y)**2 - 1.5*x + 2.5*y + 1.0



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-1.5, 4.0, -3.0, 4.0, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(McCormick):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(McCormick):
Complete Graph: 0.000306
Hypercube Q8: 0.000306
Grid: 0.000306
Grid + Diagonals: 0.000306
Grid + Main Diagonals: 0.000306
QMOA Complete: 0.000306
QMOA Complete + local Diagonals: 0.000306
QMOA Complete + Principal Complete Diagonals: 0.000306
Grid + Complete Diagonals: 0.000306
Grid with Parallel Edges: 0.000306


In [6]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    term1 = np.sin(x) * np.sin(y)
    term2 = np.exp(np.abs(100 - (np.sqrt(x**2 + y**2) / np.pi)))
    return -0.0001 * (np.abs(term1 * term2) + 1)**0.1



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-9.9, 10, -9.9, 10, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Cross-in-tray):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Cross-in-tray):
Complete Graph: 0.011590
Hypercube Q8: 0.003502
Grid: 0.004850
Grid + Diagonals: 0.007295
Grid + Main Diagonals: 0.005402
QMOA Complete: 0.016112
QMOA Complete + local Diagonals: 0.015528
QMOA Complete + Principal Complete Diagonals: 0.004009
Grid + Complete Diagonals: 0.000175
Grid with Parallel Edges: 0.004850


In [7]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    num = np.sin(x**2 - y**2)**2 - 0.5
    den = (1 + 0.001*(x**2 + y**2))**2
    return 0.5 + num / den



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-99.9, 100, -99.9, 100, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Schaffer function N. 2):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Schaffer function N. 2):
Complete Graph: 0.002490
Hypercube Q8: 0.000180
Grid: 0.002458
Grid + Diagonals: 0.002589
Grid + Main Diagonals: 0.002589
QMOA Complete: 0.002216
QMOA Complete + local Diagonals: 0.002588
QMOA Complete + Principal Complete Diagonals: 0.002584
Grid + Complete Diagonals: 0.000719
Grid with Parallel Edges: 0.002458


## Deceptive

In [8]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    a, b, c = 20, 0.2, 2 * np.pi
    x, y = np.array(x), np.array(y)
    sum_sq = 0.5 * (x ** 2 + y ** 2)
    cos_comp = 0.5 * (np.cos(c * x) + np.cos(c * y))
    return -a * np.exp(-b * np.sqrt(sum_sq)) - np.exp(cos_comp) + a + np.exp(1)



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-4.9, 5, -4.9, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Acklay)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Acklay)):
Complete Graph: 0.020871
Hypercube Q8: 0.014371
Grid: 0.004850
Grid + Diagonals: 0.009314
Grid + Main Diagonals: 0.005402
QMOA Complete: 0.029955
QMOA Complete + local Diagonals: 0.028871
QMOA Complete + Principal Complete Diagonals: 0.014313
Grid + Complete Diagonals: 0.004241
Grid with Parallel Edges: 0.004850


In [9]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return -(y + 47) * np.sin(np.sqrt(abs(x/2 + (y + 47)))) \
           - x * np.sin(np.sqrt(abs(x - (y + 47))))



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-5, 5, -5, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Eggholder)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Eggholder)):
Complete Graph: 0.011952
Hypercube Q8: 0.014110
Grid: 0.004850
Grid + Diagonals: 0.009314
Grid + Main Diagonals: 0.005402
QMOA Complete: 0.013864
QMOA Complete + local Diagonals: 0.014194
QMOA Complete + Principal Complete Diagonals: 0.002292
Grid + Complete Diagonals: 0.002153
Grid with Parallel Edges: 0.004850


In [10]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return 418.9829*2 - (x*np.sin(np.sqrt(np.abs(x))) + y*np.sin(np.sqrt(np.abs(y))))



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-5, 5, -5, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Schwefel)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Schwefel)):
Complete Graph: 0.007628
Hypercube Q8: 0.009740
Grid: 0.004850
Grid + Diagonals: 0.009314
Grid + Main Diagonals: 0.005402
QMOA Complete: 0.009656
QMOA Complete + local Diagonals: 0.009615
QMOA Complete + Principal Complete Diagonals: 0.000563
Grid + Complete Diagonals: 0.000501
Grid with Parallel Edges: 0.004850


In [11]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    w1 = 1 + (x - 1) / 4
    w2 = 1 + (y - 1) / 4
    term1 = np.sin(np.pi * w1) ** 2
    term2 = ((w1 - 1) ** 2) * (1 + 10 * np.sin(np.pi * w1 + 1) ** 2)
    term3 = ((w2 - 1) ** 2) * (1 + np.sin(2 * np.pi * w2) ** 2)
    return term1 + term2 + term3



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-5, 5, -5, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Levy N.13)):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Levy N.13)):
Complete Graph: 0.003271
Hypercube Q8: 0.003169
Grid: 0.003760
Grid + Diagonals: 0.003760
Grid + Main Diagonals: 0.002505
QMOA Complete: 0.003492
QMOA Complete + local Diagonals: 0.003482
QMOA Complete + Principal Complete Diagonals: 0.000739
Grid + Complete Diagonals: 0.000131
Grid with Parallel Edges: 0.003760


In [12]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    a = np.array([3, 5, 2, 1, 7])
    b = np.array([5, 2, 1, 4, 9])
    c = np.array([1, 2, 5, 2, 3])
    m = len(a)

    total = 0
    for i in range(m):
        r2 = (x - a[i])**2 + (y - b[i])**2
        total += c[i] * np.exp(-r2 / np.pi) * np.cos(np.pi * r2)
    return total



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,0, 10, 0, 10, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Langerman):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Langerman):
Complete Graph: 0.031827
Hypercube Q8: 0.017158
Grid: 0.000695
Grid + Diagonals: 0.000456
Grid + Main Diagonals: 0.000126
QMOA Complete: 0.042793
QMOA Complete + local Diagonals: 0.030564
QMOA Complete + Principal Complete Diagonals: 0.006623
Grid + Complete Diagonals: 0.000565
Grid with Parallel Edges: 0.000669


In [13]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return 2*x**2 - 1.05*(x**4) + (x**6)/6 + x*y + y**2



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-4.9, 5, -4.9, 5, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Three Hump Camel):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Three Hump Camel):
Complete Graph: 0.000025
Hypercube Q8: 0.000007
Grid: 0.000025
Grid + Diagonals: 0.000025
Grid + Main Diagonals: 0.000025
QMOA Complete: 0.000025
QMOA Complete + local Diagonals: 0.000025
QMOA Complete + Principal Complete Diagonals: 0.000025
Grid + Complete Diagonals: 0.000018
Grid with Parallel Edges: 0.000025


In [14]:
import numpy as np
import networkx as nx
from scipy.linalg import eigh

# -------------------------
# Spectral gap calculator
# -------------------------
rows, cols = 16, 16
n = rows * cols 
def spectral_gap(H):
    evals = np.sort(eigh(H, eigvals_only=True))
    return evals[1] - evals[0]

def interpolate_gap(L, H_C, steps=200):
    s_vals = np.linspace(0, 1, steps)
    gaps = []
    for s in s_vals:
        Hs = (1-s) * L + s * H_C
        gaps.append(spectral_gap(Hs))
    return np.min(gaps)

# -------------------------
# Example Cost Hamiltonian
# (Diagonal encoding, replace with your real cost function!)
# -------------------------
def cost_fn(x, y):
    return -np.cos(x) * np.cos(y) * np.exp(-((x - np.pi)**2 + (y - np.pi)**2))



def build_cost_hamiltonian(cost_func, x_min, x_max, y_min, y_max, rows, cols):
    dx = (x_max - x_min) / (rows - 1)
    dy = (y_max - y_min) / (cols - 1)

    costs = []
    for i in range(rows):
        for j in range(cols):
            x = x_min + i * dx
            y = y_min + j * dy
            costs.append(cost_func(x, y))

    H_C = np.diag(costs)

    # --- Normalize H_C to [0, 1] ---
    E_min, E_max = np.min(costs), np.max(costs)
    H_C = (H_C - E_min * np.eye(rows*cols)) / (E_max - E_min)

    return H_C

def build_grid(rows, cols):
    return nx.convert_node_labels_to_integers(nx.grid_2d_graph(rows, cols))

def build_grid_with_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))
            G.add_edge((i + 1, j), (i, j + 1))
    return nx.convert_node_labels_to_integers(G)

def build_grid_with_main_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)
    for i in range(min(rows - 1, cols - 1)):
        G.add_edge((i, i), (i + 1, i + 1))
        G.add_edge((i, cols - 1 - i), (i + 1, cols - 2 - i))
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_complete(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    return nx.convert_node_labels_to_integers(G)

def build_qmoa_with_diagonals(rows, cols):
    G = nx.Graph()
    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))
    # All-to-all in rows
    for r in range(rows):
        row_nodes = [(r, c) for c in range(cols)]
        for i in range(len(row_nodes)):
            for j in range(i + 1, len(row_nodes)):
                G.add_edge(row_nodes[i], row_nodes[j])
    # All-to-all in columns
    for c in range(cols):
        col_nodes = [(r, c) for r in range(rows)]
        for i in range(len(col_nodes)):
            for j in range(i + 1, len(col_nodes)):
                G.add_edge(col_nodes[i], col_nodes[j])
    # Add diagonals
    for i in range(rows - 1):
        for j in range(cols - 1):
            G.add_edge((i, j), (i + 1, j + 1))  # Forward diagonal
            G.add_edge((i + 1, j), (i, j + 1))  # Backward diagonal

    # Relabel nodes to 0...n-1 to match H_C
    return nx.convert_node_labels_to_integers(G, ordering="sorted")
def build_qmoa_with_principal_complete_diagonals(rows, cols):
    G = nx.Graph()

    for r in range(rows):
        for c in range(cols):
            G.add_node((r, c))

    # Row-wise complete connections
    for r in range(rows):
        row = [(r, c) for c in range(cols)]
        for i in range(len(row)):
            for j in range(i + 1, len(row)):
                G.add_edge(row[i], row[j])

    # Column-wise complete connections
    for c in range(cols):
        col = [(r, c) for r in range(rows)]
        for i in range(len(col)):
            for j in range(i + 1, len(col)):
                G.add_edge(col[i], col[j])

    # Principal main diagonal (↘)
    main_diag = [(i, i) for i in range(min(rows, cols))]
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Principal anti-diagonal (↙)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_grid_with_complete_diagonals(rows, cols):
    G = nx.grid_2d_graph(rows, cols)

    # Collect nodes along the main diagonal (top-left to bottom-right)
    main_diag = [(i, i) for i in range(min(rows, cols))]

    # Collect nodes along the anti-diagonal (top-right to bottom-left)
    anti_diag = [(i, cols - 1 - i) for i in range(min(rows, cols))]

    # Add complete edges among all nodes on main diagonal
    for i in range(len(main_diag)):
        for j in range(i + 1, len(main_diag)):
            G.add_edge(main_diag[i], main_diag[j])

    # Add complete edges among all nodes on anti-diagonal
    for i in range(len(anti_diag)):
        for j in range(i + 1, len(anti_diag)):
            G.add_edge(anti_diag[i], anti_diag[j])

    return nx.convert_node_labels_to_integers(G)

def build_multigraph_grid(rows, cols, edge_multiplicity=2):
    G = nx.MultiGraph()
    for i in range(rows):
        for j in range(cols):
            if j < cols - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i, j + 1))
            if i < rows - 1:
                for _ in range(edge_multiplicity):
                    G.add_edge((i, j), (i + 1, j))
    return G

def multigraph_to_weighted_graph(G_multi):
    G_weighted = nx.Graph()
    for u, v in G_multi.edges():
        if G_weighted.has_edge(u, v):
            G_weighted[u][v]['weight'] += 1
        else:
            G_weighted.add_edge(u, v, weight=1)
    return nx.convert_node_labels_to_integers(G_weighted)

# -------------------------
# Graph Laplacians
# -------------------------
def build_laplacians(n):
    graphs = {
    "Complete Graph": nx.complete_graph(n),
    "Hypercube Q8": nx.convert_node_labels_to_integers(nx.hypercube_graph(8)),
    "Grid": build_grid(rows, cols),
    "Grid + Diagonals": build_grid_with_diagonals(rows, cols),
    "Grid + Main Diagonals": build_grid_with_main_diagonals(rows, cols),
    "QMOA Complete": build_qmoa_complete(rows, cols),
    "QMOA Complete + local Diagonals": build_qmoa_with_diagonals(rows, cols),
    "QMOA Complete + Principal Complete Diagonals": build_qmoa_with_principal_complete_diagonals(rows, cols),
    "Grid + Complete Diagonals" : build_grid_with_complete_diagonals(rows, cols) }
    # Add weighted version of multiedge grid
    G_multi_raw = build_multigraph_grid(rows, cols, edge_multiplicity=2)
    G_multi_weighted = multigraph_to_weighted_graph(G_multi_raw)
    graphs["Grid with Parallel Edges"] = G_multi_weighted

    Ls = {}
    for name, G in graphs.items():
        if G is not None:
            L = nx.laplacian_matrix(G).toarray()

            # --- Normalize Laplacian to [0,1] ---
            eigvals = np.linalg.eigvalsh(L)
            L = (L - eigvals[0] * np.eye(L.shape[0])) / (eigvals[-1] - eigvals[0])

            Ls[name] = L
    return Ls

# -------------------------
# Run comparison
# -------------------------
def compare_gaps():
    H_C = build_cost_hamiltonian(cost_fn,-7*np.pi, 8*np.pi,-7*np.pi, 8*np.pi, rows, cols)
    Ls = build_laplacians(n)
    results = {}
    for name, L in Ls.items():
        Δmin = interpolate_gap(L, H_C)
        results[name] = Δmin
    return results


# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    results = compare_gaps()
    print("Minimum Spectral Gaps(Easom):")
    for graph, gap in results.items():
        print(f"{graph}: {gap:.6f}")



Minimum Spectral Gaps(Easom):
Complete Graph: 0.062697
Hypercube Q8: 0.032533
Grid: 0.004833
Grid + Diagonals: 0.008845
Grid + Main Diagonals: 0.005348
QMOA Complete: 0.056690
QMOA Complete + local Diagonals: 0.054788
QMOA Complete + Principal Complete Diagonals: 0.057258
Grid + Complete Diagonals: 0.006568
Grid with Parallel Edges: 0.004833
